In [1]:
# # 🌾 AgriIntel

# ## Phase 5

# # Master Dataset Integration

# ### Objective

# The objective of this phase is to integrate multiple agricultural datasets into
# one unified analytical dataset.

# Datasets:

# • Crop Production

# • Crop Yield

# • Weather

# • Soil

# The final output of this notebook will be

# master_dataset.csv

In [2]:
# ============================================================
# AgriIntel
#
# Phase 5
#
# Master Dataset Integration
# ============================================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns",None)

In [3]:
crop = pd.read_csv("../data/processed/crop_production_clean.csv")

yield_df = pd.read_csv("../data/processed/crop_yield_clean.csv")

weather = pd.read_csv("../data/processed/state_weather_clean.csv")

soil = pd.read_csv("../data/processed/state_soil_clean.csv")

In [4]:
yield_df.rename(columns={

"crop":"Crop",

"state":"State",

"season":"Season",

"year":"Year",

"area":"Area",

"production":"Production",

"yield":"Yield"

},inplace=True)

weather.rename(columns={

"state":"State",

"year":"Year"

},inplace=True)

soil.rename(columns={

"state":"State"

},inplace=True)

In [5]:
crop["Start_Year"]=(
crop["Year"]
.str.split("-")
.str[0]
.astype(int)
)

In [6]:
crop_yield = pd.merge(

crop,

yield_df,

left_on=[
"State",
"Crop",
"Season",
"Start_Year"
],

right_on=[
"State",
"Crop",
"Season",
"Year"
],

how="left",

suffixes=(
"",
"_yield"
),

indicator=True

)

In [7]:
crop_yield["_merge"].value_counts()

_merge
both          318023
left_only      27351
right_only         0
Name: count, dtype: int64

In [8]:
success=(
crop_yield["_merge"]
.eq("both")
.mean()
*100
)

print(success)

92.08075882955869


In [9]:
crop_yield[
crop_yield["_merge"]=="left_only"
].head()

,State,District,Crop,Year,Season,Area,Area Units,Production,Production Units,Yield,Start_Year,Year_yield,Area_yield,Production_yield,fertilizer,pesticide,Yield_yield,_merge
0,Andaman And Nicobar Islands,Nicobars,Arecanut,2001-02,Kharif,1254.0,Hectare,2061.0,Tonnes,1.643541,2001,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,Andaman And Nicobar Islands,Nicobars,Arecanut,2002-03,Whole Year,1258.0,Hectare,2083.0,Tonnes,1.655803,2002,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,Andaman And Nicobar Islands,Nicobars,Arecanut,2003-04,Whole Year,1261.0,Hectare,1525.0,Tonnes,1.209358,2003,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,Andaman And Nicobar Islands,North And Middle Andaman,Arecanut,2001-02,Kharif,3100.0,Hectare,5239.0,Tonnes,1.690000,2001,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,Andaman And Nicobar Islands,South Andamans,Arecanut,2002-03,Whole Year,3105.0,Hectare,5267.0,Tonnes,1.696296,2002,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [10]:
crop_yield.drop(
columns="_merge",
inplace=True
)

In [11]:
master = pd.merge(

crop_yield,

weather,

left_on=[
"State",
"Start_Year"
],

right_on=[
"State",
"Year"
],

how="left",

suffixes=(
"",
"_weather"
),

indicator=True

)

In [12]:
master["_merge"].value_counts()

_merge
both          323787
left_only      21587
right_only         0
Name: count, dtype: int64

In [13]:
master.drop(
columns="_merge",
inplace=True
)

In [14]:
master = pd.merge(

master,

soil,

on="State",

how="left",

indicator=True

)

In [15]:
master["_merge"].value_counts()

_merge
both          323787
left_only      21587
right_only         0
Name: count, dtype: int64

In [16]:
master.drop(
columns="_merge",
inplace=True
)

In [17]:
master.shape

(345374, 25)

In [18]:
master.isnull().sum()

State                       0
District                    0
Crop                        0
Year                        0
Season                      0
Area                        0
Area Units                  0
Production               4960
Production Units            0
Yield                       0
Start_Year                  0
Year_yield              27351
Area_yield              27351
Production_yield        27351
fertilizer              27351
pesticide               27351
Yield_yield             27351
Year_weather            21587
avg_temp_c              21587
total_rainfall_mm       21587
avg_humidity_percent    21587
N                       21587
P                       21587
K                       21587
pH                      21587
dtype: int64

In [19]:
master.head()

,State,District,Crop,Year,Season,Area,Area Units,Production,Production Units,Yield,Start_Year,Year_yield,Area_yield,Production_yield,fertilizer,pesticide,Yield_yield,Year_weather,avg_temp_c,total_rainfall_mm,avg_humidity_percent,N,P,K,pH
0,Andaman And Nicobar Islands,Nicobars,Arecanut,2001-02,Kharif,1254.0,Hectare,2061.0,Tonnes,1.643541,2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Andaman And Nicobar Islands,Nicobars,Arecanut,2002-03,Whole Year,1258.0,Hectare,2083.0,Tonnes,1.655803,2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Andaman And Nicobar Islands,Nicobars,Arecanut,2003-04,Whole Year,1261.0,Hectare,1525.0,Tonnes,1.209358,2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Andaman And Nicobar Islands,North And Middle Andaman,Arecanut,2001-02,Kharif,3100.0,Hectare,5239.0,Tonnes,1.690000,2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Andaman And Nicobar Islands,South Andamans,Arecanut,2002-03,Whole Year,3105.0,Hectare,5267.0,Tonnes,1.696296,2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
master.columns.tolist()

['State',
 'District',
 'Crop',
 'Year',
 'Season',
 'Area',
 'Area Units',
 'Production',
 'Production Units',
 'Yield',
 'Start_Year',
 'Year_yield',
 'Area_yield',
 'Production_yield',
 'fertilizer',
 'pesticide',
 'Yield_yield',
 'Year_weather',
 'avg_temp_c',
 'total_rainfall_mm',
 'avg_humidity_percent',
 'N',
 'P',
 'K',
 'pH']

In [21]:
master.to_csv(

"../data/merged/master_dataset.csv",

index=False

)

In [22]:
report=pd.DataFrame({

"Dataset":[

"Crop",

"Yield",

"Weather",

"Soil"

],

"Rows":[

len(crop),

len(yield_df),

len(weather),

len(soil)

],

"Status":[

"Integrated",

"Integrated",

"Integrated",

"Integrated"

]

})

report

,Dataset,Rows,Status
0,Crop,345374,Integrated
1,Yield,19689,Integrated
2,Weather,720,Integrated
3,Soil,30,Integrated


In [24]:
# # Phase 5 Conclusion

# The four agricultural datasets were successfully integrated into one
# master dataset.

# The integrated dataset now contains

# • Crop Information

# • Yield

# • Weather

# • Soil

# This dataset will be used for Feature Engineering and Advanced Analytics.

In [25]:
print("="*60)
print("PHASE 5 VERIFICATION")
print("="*60)

print("\nMaster Shape")
print(master.shape)

print("\nColumns")
print(master.columns.tolist())

print("\nMissing Values")
print(master.isnull().sum())

print("\nFirst 5 Rows")
print(master.head())

print("\nMerge Report")
print(report)

PHASE 5 VERIFICATION

Master Shape
(345374, 25)

Columns
['State', 'District', 'Crop', 'Year', 'Season', 'Area', 'Area Units', 'Production', 'Production Units', 'Yield', 'Start_Year', 'Year_yield', 'Area_yield', 'Production_yield', 'fertilizer', 'pesticide', 'Yield_yield', 'Year_weather', 'avg_temp_c', 'total_rainfall_mm', 'avg_humidity_percent', 'N', 'P', 'K', 'pH']

Missing Values
State                       0
District                    0
Crop                        0
Year                        0
Season                      0
Area                        0
Area Units                  0
Production               4960
Production Units            0
Yield                       0
Start_Year                  0
Year_yield              27351
Area_yield              27351
Production_yield        27351
fertilizer              27351
pesticide               27351
Yield_yield             27351
Year_weather            21587
avg_temp_c              21587
total_rainfall_mm       21587
avg_humidity_p

In [34]:
crop_yield.shape

(345374, 17)

In [35]:
master.shape

(345374, 25)